In [ ]:
import nibabel as nib
import numpy as np
from pathlib import Path

data_dir = Path("data/Merlin Abdominal CT Dataset_files")
files = [f for f in sorted(data_dir.glob("*.nii.gz"))]

print(files)
# img = nib.load(files[0])
# volume = img.get_fdata()

# print(volume)
# print(volume.shape)
# print(volume.dtype)

# print(volume.min())
# print(volume.max())

# img.header.get_zooms()

In [24]:
FILTERED_FINDING_PHRASES = {
    "renal_cyst":                     ("a renal cyst is present", "no renal cyst"),
    "surgically_absent_gallbladder":  ("the gallbladder is surgically absent", "the gallbladder is present"),
    "atelectasis":                    ("atelectasis is present", "no atelectasis"),
    "pleural_effusion":               ("a pleural effusion is present", "no pleural effusion"),
}

In [77]:
from merlin.data import DataLoader
from merlin import Merlin
import torch
import torch.nn.functional as F
import json
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"

datalist = [{"image": f, "study_id": f.stem.removesuffix(".nii")} for f in files]
# datalist = [{"image": files[0], "study_id": "AC421363e"}, {"image": files[1], "study_id": "AC421363f"}]

dataloader = DataLoader(
    datalist=datalist,
    cache_dir=None,
    batchsize=1,
    shuffle=True,
    num_workers=0,
)

# model_img = Merlin(ImageEmbedding=True)
# model_img.eval().cuda()

model = Merlin()
model.eval().cuda()

flat_phrases, phrase_keys = [], []

for finding, (present, absence) in FILTERED_FINDING_PHRASES.items():
    flat_phrases += [present, absence]
    phrase_keys += [(finding, "present"), (finding, "absent")]

img_embeds = {}
dummy_tensor = None
with torch.no_grad():
    for batch in dataloader:
        sid = batch["study_id"][0]
        if dummy_tensor is None:
            dummy_tensor = batch["image"].cuda()
        img_out, _, _ = model(batch["image"].cuda(), flat_phrases)
        img_embeds[sid] = img_out[0]

with torch.no_grad():
    _, _, text_embeds = model(dummy_tensor, flat_phrases)

text_lookup = {}
for i, (finding, polarity) in enumerate(phrase_keys):
    text_lookup.setdefault(finding, {})[polarity] = text_embeds[i]

labels = pd.read_csv("data/zero_shot_findings_disease_cls.csv").set_index("study_id")

results = {}

with torch.no_grad():
    for finding in FILTERED_FINDING_PHRASES:
        sid = list(img_embeds.keys())

        sid_list, y_true, y_score = [], [], []

        for sid_i in sid:
            label = labels.loc[sid_i, finding]
            if label == -1:
                continue

            sim_present = F.cosine_similarity(img_embeds[sid_i], text_lookup[finding]["present"], dim=0)
            sim_absent = F.cosine_similarity(img_embeds[sid_i], text_lookup[finding]["absent"], dim=0)

            probs = F.softmax(torch.stack([sim_present, sim_absent]), dim=0)
            score = probs[0].item()

            sid_list.append(sid_i)
            y_score.append(score)
            y_true.append(int(label))

        results[finding] = {"study_id": sid_list, "y_true": y_true, "y_score": y_score}

        
        with open("merlin_zero_shot_results.json", "w") as f:
            json.dump(results, f, indent=2)

Size of dataset: 100



/home/rahuldeb5/Cancer-Detection/.venv/lib/python3.14/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/rahuldeb5/Cancer-Detection/.venv/lib/python3.14/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet152_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet152_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Loading weights: 100%|██████████| 269/269 [00:00<00:00, 9709.62it/s]
[transformers] LongformerModel LOAD REPORT from: yikuan8/Clinical-Longformer
Key                                | Status     | 
-----------------------------------+------------+-
longformer.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.b

Loading checkpoint for 'default' task from /home/rahuldeb5/Cancer-Detection/.venv/lib/python3.14/site-packages/merlin/models/checkpoints/i3_resnet_clinical_longformer_best_clip_04-02-2024_23-21-36_epoch_99.pt


/home/rahuldeb5/Cancer-Detection/.venv/lib/python3.14/site-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/home/rahuldeb5/Cancer-Detection/.venv/lib/python3.14/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
